# Lesson 8: Use the controller's get_ and set_ API

This notebook teaches developers how to use `VerdiController`: **read values with
`get_...`, change settings with `set_...`, and check the result**. Prefer named
methods such as `get_power_w()` and `set_power_w()` when available. Use
`get_value()` and `set_value()` for the general text interface.

| Part | Sections | What you will learn |
| --- | --- | --- |
| [Connect](#connect) | 1-4 | Create a controller and start a session. |
| [Get values](#get-api) | 5-8 | Use every available `get_...` method and understand its return value. |
| [Set values](#set-api) | 9-13 | Use every available `set_...` method, its arguments and relevant readback. |
| [Handle errors and finish](#finish) | 14-15 | Inspect an error and release the connection. |
| [Supporting reference](#reference) | A-D | Look up queries, byte transport, exceptions and context management. |

**How to use this notebook:** run executable cells one at a time with Shift+Enter.
Every public controller method has an executable code cell. By default, cells
use real hardware for queries only. To test a write, open the optional writer in
section 9, then change `RUN_THIS_EXAMPLE` to `True` in only the desired command
cell. Supply any approved values in that cell and inspect the result. Use [lesson 5](05_set_power.ipynb) or [lesson 6](06_key_on_and_run.ipynb) for
complete operator-controlled write procedures.

Keep one serial owner: close other kernels, terminals or dashboards using this
port. Restart this kernel after package updates. If a communication/parsing cell
fails, stop, inspect section 14, then run **Disconnect** in section 15. Do not
purge or retry after an error. Serial disconnection is not an emergency stop;
use the operator's physical procedure when needed.

See the [API documentation](../docs/API.md), [hardware guide](../docs/HARDWARE.md)
and [operator manual](../verdi.manual_v5.pdf) for the complete contracts.


<a id="connect"></a>
# Connect

The lifecycle is: **create -> connect -> query or set -> disconnect**.
Construction alone opens nothing and sends nothing.


## 1. Enter your connection settings

Use [lesson 0](00_device_discovery.ipynb) to identify the port. Confirm the model,
baud rate and enabled prompt before editing this cell.

| Setting | Meaning |
| --- | --- |
| `PORT` | The native serial port name, such as `"COM5"`. Replace `"<PORT>"`. |
| `MODEL` | `"V2"`, `"V5"` or `"V6"`, matching the actual laser. |
| `BAUDRATE` | Communication speed; it must match the front-panel setting. |
| `TIMEOUT_S` | Serial write and receive timeout; reply lines, notices and prompt are read separately. |

The serial format is 8 data bits, no parity and 1 stop bit (8N1), with no flow
control. The controller does not discover the port or baud rate automatically.


In [1]:
PORT = "COM5"  # Enter the port found in lesson 0, e.g. "COM5".
MODEL = "V2"  # V2, V5 or V6: match the actual laser.
BAUDRATE = 19200  # Match the front panel.
TIMEOUT_S = 2.0

## 2. Create a controller

`laser` is the object used throughout the executable examples. Writes are disabled
explicitly with `allow_writes=False`. `prompt_enabled=True` assumes the instrument
prompt is already enabled; construction does not change it.

`pprint()` displays dictionaries with readable spacing. `power_limit_w` defaults
to the model rating; a write session needs a site-approved operating limit.


In [2]:
from pprint import pprint

from coherent_verdi import VerdiController

if not isinstance(PORT, str) or not PORT.strip() or PORT == "<PORT>":
    raise ValueError("Edit the connection settings in section 1 first.")

laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    allow_writes=False,
    prompt_enabled=True,
)
writer = None  # Optional write session; also available to error/cleanup cells.

In [3]:
print("Model:", laser.model)
print("Simulated:", laser.is_simulated)
print("Writes allowed:", laser.allow_writes)
print("Power ceiling:", laser.power_limit_w, "W")
print("Session failed:", laser.failed)
print("Previous transaction:", laser.last_transaction)

Model: V2
Simulated: False
Writes allowed: False
Power ceiling: 2.0 W
Session failed: False
Previous transaction: None


## 3. Open the serial port

Run this cell separately and inspect the result. `True` confirms the local port
opened; it does not confirm a laser reply or instrument readiness. Opening errors raise.

`override=True` closes an older controller-owned handle to this port in the same
Python process. Use `override=False` to refuse replacement. Neither can take the
port from another kernel, application or raw serial handle; close that owner first.


In [4]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

Serial port open: True


## 4. Start communication with get_value()

First clear the host queues once, before the first query. `purge_buffers()` discards
pending outgoing bytes (TX), then incoming bytes (RX), and sends no instrument
instruction. It does not reset the laser's parser or repair a failed session.


In [5]:
laser.purge_buffers()  # Once, before the first query; explicit host queue discard.

### Ask for the software version

`get_value("?SV")` sends the software-version query and returns its interpreted
value. For an ASCII version reply, the result is Python text (`str`). No manual
byte decoding is needed. Non-ASCII text remains `bytes` rather than guessing an encoding.


In [6]:
version = laser.get_value("?SV")
print("Software version:", repr(version))

Software version: '6.05'


<a id="get-api"></a>
# Get values: the get_ API

Every getter performs fresh communication; these are methods, not cached
properties. Consecutive readings can differ because they are acquired at different times.

| Available method | Return value | Use it for |
| --- | --- | --- |
| `get_power_w()` | `float` in W | Measured optical power |
| `get_laser_state()` | `int` code | Standby, on or fault-off |
| `get_faults(history=False)` | `list[int]` | Active faults or fault history |
| `get_value(query)` | Number, text, bytes or fault-code list | Any catalogued query |
| `get_status()` | `dict` | Operating measurements and states |
| `get_diagnostics()` | `dict` | Version, operating hours and fault history |
| `get_capture(query)` | Transaction `dict` | Byte evidence without value interpretation; read-only sessions only |


## 5. Use the named getters

### 5.1 get_power_w(): measured power

The `_w` suffix specifies watts. Store the returned number to use it in calculations.
Measured output is different from the stored setpoint, read with `get_value("?SP")`.


In [7]:
power_w = laser.get_power_w()
print("Measured power:", power_w, "W")

Measured power: 0.025 W


### 5.2 get_laser_state(): operating state

The return value is an integer: `0` = standby, `1` = on, `2` = fault-off.
The shutter position is a separate query. A closed shutter does not prove the laser is off.


In [8]:
state = laser.get_laser_state()
state_labels = {0: "standby", 1: "on", 2: "fault-off"}
print("Laser state:", state, state_labels[state])

Laser state: 1 on


### 5.3 get_faults(): current faults and history

Without arguments, this reads active faults. The keyword argument `history=True`
reads fault history instead. Reading either does not clear it.

`SYSTEM OK` becomes an empty list `[]`. Positive codes remain in the list,
including unknown codes; unexpected replies raise `VerdiError`.


In [9]:
active_faults = laser.get_faults()
fault_history = laser.get_faults(history=True)
print("Active faults:", active_faults)
print("Fault history:", fault_history)

Active faults: []
Fault history: []


## 6. Use get_value() for the remaining queries

There is no dedicated getter for every measurement. Pass a documented query as a
Python string to `get_value()`; the controller converts its reply to the appropriate
Python type. For example, `get_value("?S")` reads the shutter state.

| Example | Meaning | Returned value |
| --- | --- | --- |
| `get_value("?SP")` | Stored power setpoint | `float`, W |
| `get_value("?S")` | Safety shutter | `int`: 0 closed, 1 open |
| `get_value("?K")` | Physical keyswitch | `int`: 0 off, 1 on |
| `get_value("?D1C")` | Diode current | `float`, A |
| `get_value("?D1T")` | Diode temperature | `float`, degrees C |
| `get_value("?LBOSS")` | LBO servo state | `int` status code |
| `get_value("?SV")` | Software version | ASCII `str`, or undecoded non-ASCII `bytes` |

Appendix A shows how to look up all query names, units and code labels. Looking
up a query does not send it; do not assume every firmware supports every query.


In [10]:
setpoint_w = laser.get_value("?SP")
shutter_code = laser.get_value("?S")
diode_temp_c = laser.get_value("?D1T")
print("Stored power:", setpoint_w, "W")
print("Shutter code:", shutter_code)
print("Diode temperature:", diode_temp_c, "degrees C")

Stored power: 0.01 W
Shutter code: 0
Diode temperature: 27.88 degrees C


### Optional: long-name aliases

`get_value("PRINT LIGHT")` and `get_value("?LIGHT")` both send `?P`, just like
`get_value("?P")` or `get_power_w()`. Named helpers are usually clearest when available.


In [11]:
print("PRINT LIGHT:", laser.get_value("PRINT LIGHT"))
print("?LIGHT:", laser.get_value("?LIGHT"))

PRINT LIGHT: 0.025
?LIGHT: 0.025


## 7. Get related values as dictionaries

### 7.1 get_status(): operating snapshot

Use dictionary keys to access the fields you need. `get_status()` makes 14
sequential queries and adds `sampled_at` (UTC start time), `duration_s`, `model`
and `simulated`. The readings are not simultaneous.

Individual getters return integer state codes; this status dictionary converts
`shutter_open` and `keyswitch_on` to Python booleans.


In [12]:
status = laser.get_status()
pprint(status)
print("Measured power:", status["power_w"], "W")
print("Stored power:", status["set_power_w"], "W")
print("Shutter open:", status["shutter_open"])

{'baseplate_temp_c': 26.6,
 'diode_current_a': 11.84,
 'diode_temp_c': 27.88,
 'duration_s': 0.5,
 'etalon_temp_c': 70.02,
 'faults': [],
 'heatsink_temp_c': 25.13,
 'keyswitch_on': True,
 'laser_state': 1,
 'lbo_servo': 1,
 'lbo_temp_c': 149.24,
 'model': 'V2',
 'power_w': 0.025,
 'sampled_at': '2026-10-27T23:10:24.191783+00:00',
 'set_power_w': 0.01,
 'shutter_open': False,
 'simulated': False,
 'vanadate_temp_c': 35.04}
Measured power: 0.025 W
Stored power: 0.01 W
Shutter open: False


### 7.2 get_diagnostics(): instrument background

This dictionary contains software version, head/power-supply/diode operating
hours and fault history. Returned dictionaries and lists are independent copies.


In [13]:
diagnostics = laser.get_diagnostics()
pprint(diagnostics)
print("Head operating hours:", diagnostics["head_hours"])

{'diode_hours': 2390123.54,
 'fault_history': [],
 'head_hours': 2694.91,
 'power_supply_hours': 2694.91,
 'software_version': '6.05'}
Head operating hours: 2694.91


## 8. Use get_capture() when you need byte evidence

Ordinary application code should use the interpreted getters above. Use capture
when you need to inspect a documented query's payload before assigning it a meaning.
`get_capture()` requires `allow_writes=False` and returns a transaction dictionary;
its `payload` is always bytes. It still validates framing and device rejections.

An empty or clear-looking byte payload alone does not establish no faults or
readiness. This is a fresh query, not a conversion of section 5's result.


In [14]:
fault_capture = laser.get_capture("?F")
print("Active-fault payload:", repr(fault_capture["payload"]))
pprint(fault_capture)

Active-fault payload: b'SYSTEM OK'
{'error': None,
 'instruction': b'?F',
 'outcome': 'captured',
 'payload': b'SYSTEM OK',
 'request': b'?F\r\n',
 'response': b'SYSTEM OK\r\nVerdi V-2>'}


In [15]:
# read(): payload bytes; command(): complete wire bytes (Appendix B).
payload = laser.read(b"?P")
print("Power payload:", repr(payload))
wire_reply = laser.command(b"?SV")
print("Version wire reply:", repr(wire_reply))

Power payload: b'0.025'
Version wire reply: b'6.05\r\nVerdi V-2>'


<a id="set-api"></a>
# Set values: the set_ API

Every setter below has its own executable code cell, including relevant readback.
Each starts with `RUN_THIS_EXAMPLE = False`: running it prints a skip message.
After preparing the optional `writer` in section 9, change that flag to `True`
only in the cell you intend to test. Set it back to `False` afterwards.

These are independent API examples, not an operating sequence. Establish the
conditions for the chosen action with the operator; lessons 5-6 describe complete
power/emission procedures. Do not enable all examples and run them as a batch.


## 9. Understand the setter contract

A setter validates its argument, sends the instruction and validates the
acknowledgment. **A successful setter returns `None`**; it does not return the new
measurement or wait for physical settling. Read the relevant state or measurement
with a getter afterwards. An acknowledgment establishes protocol acceptance only.

The optional writer uses the same connection settings as the reader. Enter a
site-approved power ceiling and set `OPEN_WRITE_SESSION = True` in the setup cell
below. Values left as `None` must be filled in before enabling their example.
The setup releases the read-only `laser` before opening `writer`; inspect the
connection result before running the separate communication cell.

After opening the writer, the earlier `laser` query cells are disconnected.
Finish with section 15 before starting another session. A prompt or baud change
also changes the settings needed for the next connection; record the accepted
values before disconnecting.

### Check local write protection

This executable demonstration checks that our `laser` is read-only, then attempts
a setting that must be refused with `PermissionError` before any instrument
instruction is sent. The previous transaction must remain unchanged.

Saved hardware outputs are retained from the earlier operator run. The optional
write-session and command flags have been reset to `False` for the next run;
approved operands are retained. These outputs are not a rerun of the revised cells.


In [16]:
assert laser.allow_writes is False, "This demonstration requires a read-only controller"
before = laser.last_transaction
try:
    laser.set_power_w(0.0)
except PermissionError as error:
    print("Expected read-only refusal:", error)
else:
    raise AssertionError("The read-only controller must reject writes")
assert laser.last_transaction == before

Expected read-only refusal: writes disabled; pass allow_writes=True explicitly


In [17]:
OPEN_WRITE_SESSION = False  # Enable only after preparing the chosen action.
APPROVED_LIMIT_W = 0.1  # Enter the site-approved ceiling in W.

if OPEN_WRITE_SESSION:
    if APPROVED_LIMIT_W is None:
        raise ValueError("Enter APPROVED_LIMIT_W before opening the writer.")
    writer = VerdiController(
        PORT,
        model=MODEL,
        baudrate=BAUDRATE,
        timeout_s=TIMEOUT_S,
        prompt_enabled=True,
        allow_writes=True,
        power_limit_w=APPROVED_LIMIT_W,
    )
    laser.disconnect()
    print("Writer port open:", writer.connect(override=False))
else:
    print("Write session skipped; laser remains read-only.")

Writer port open: True


In [18]:
# Run separately after inspecting the writer's connection result.
if writer is not None:
    writer.purge_buffers()  # Once before this session's first query.
    print("Writer software version:", repr(writer.get_value("?SV")))
else:
    print("No write session is open.")

Writer software version: '6.05'


## 10. Set power, laser state and shutter

### 10.1 set_power_w(): store the power setpoint

Use watts in **0.01 W increments**, within the configured power ceiling. Invalid
values raise `ValueError` before I/O; the method does not silently round a finer target.
Setting power does not implicitly enable the laser or open the shutter.


Check the stored reading against the intended setpoint. The measured reading
is the laser's output report; it need not equal that setpoint immediately or with the shutter closed.

### General alternative: set_value()

`set_value()` accepts a canonical text instruction with `=`. Prefer the named
helper when one exists, because its arguments express the intended quantity clearly.
Choose one of the independent examples below for the same setting.


`set_value()` also validates the acknowledgment and returns `None`. Text writes
use canonical short names, not the query aliases accepted by `get_value()`.


In [19]:
# set_power_w: store power
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.
APPROVED_W = 0.01

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_power_w(APPROVED_W)
    print("Stored:", writer.get_value("?SP"), "W")
    print("Measured:", writer.get_power_w(), "W")
else:
    print("Skipped: set_power_w: store power")

Stored: 0.01 W
Measured: 0.025 W


In [20]:
# set_value: canonical text setting
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.
APPROVED_W = 0.01

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_value(f"P={APPROVED_W:.4f}")
    print("Stored:", writer.get_value("?SP"), "W")
else:
    print("Skipped: set_value: canonical text setting")

Stored: 0.01 W


In [21]:
# write: byte setting with raw acknowledgment
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.
APPROVED_W = 0.01

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    wire_reply = writer.write(f"P={APPROVED_W:.4f}".encode("ascii"))
    print("Wire acknowledgment:", repr(wire_reply))
    print("Stored:", writer.get_value("?SP"), "W")
else:
    print("Skipped: write: byte setting with raw acknowledgment")

Wire acknowledgment: b'\r\nVerdi V-2>'
Stored: 0.01 W


In [22]:
# command: byte setting with raw acknowledgment
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    wire_reply = writer.command(b"S=0")
    print("Wire acknowledgment:", repr(wire_reply))
    print("Shutter code:", writer.get_value("?S"))
else:
    print("Skipped: command: byte setting with raw acknowledgment")

Wire acknowledgment: b'\r\nVerdi V-2>'
Shutter code: 0


### 10.2 set_laser_state(): enable/reset or standby

| Independent call | Meaning | Readback |
| --- | --- | --- |
| `writer.set_laser_state(1)` | Enable/reset faults and clear fault history; requires physical key ON and complete readiness checks | `writer.get_laser_state()` |
| `writer.set_laser_state(0)` | Enter remote standby; not the full heater/cooling shutdown procedure | `writer.get_laser_state()` |

Only integer `0` or `1` can be set. State `2` is a fault-off query result, not a
settable state. Preserve active faults and history before an enable/reset.
Turning the physical key can itself enable the laser; there is no remote keyswitch setter.

### 10.3 set_shutter(): operate the safety shutter

| Independent call | Meaning | Readback |
| --- | --- | --- |
| `writer.set_shutter(open=True)` | Open the safety shutter under the operator's procedure | `writer.get_value("?S")` should report `1` |
| `writer.set_shutter(open=False)` | Close the safety shutter | `writer.get_value("?S")` should report `0` |

`open` is a keyword-only Boolean argument: use `open=True` or `open=False`.
Closing this shutter does not put the laser in standby. Never use it as an
experimental modulator. Use lesson 6 for the complete controlled sequence.


In [23]:
# set_laser_state: enable/reset
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    print("Active faults before reset:", writer.get_faults())
    print("History before reset:", writer.get_faults(history=True))
    writer.set_laser_state(1)
    print("Laser state:", writer.get_laser_state())
else:
    print("Skipped: set_laser_state: enable/reset")

Active faults before reset: []
History before reset: []
Laser state: 1


In [24]:
# set_laser_state: standby
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_laser_state(0)
    print("Laser state:", writer.get_laser_state())
else:
    print("Skipped: set_laser_state: standby")

Laser state: 0


In [25]:
# set_shutter: open
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_shutter(open=True)
    print("Shutter code:", writer.get_value("?S"))
else:
    print("Skipped: set_shutter: open")

Shutter code: 1


In [26]:
# set_shutter: close
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_shutter(open=False)
    print("Shutter code:", writer.get_value("?S"))
else:
    print("Skipped: set_shutter: close")

VerdiError: prompt timed out or unexpected: b'prom#1 hours=2390123.652049\r\nincr = 0.111185\r\nprom#3 hours=2695.019283\r\n\r\nVerdi V-2>'; full response: b'level0 set!\r\nincr = 0.111174\r\nprom#1 hours=2390123.652049\r\nincr = 0.111185\r\nprom#3 hours=2695.019283\r\n\r\nVerdi V-2>'

## 11. Thermal, optical and optional chiller setters

These methods initiate an action or store a setting. They return after acknowledgment,
not after heating, cooling, optimization or other physical completion.

| Independent call | Purpose | What to inspect next |
| --- | --- | --- |
| `writer.set_lbo_heater(enabled=True)` | Start LBO heating | `get_value("?LBOH")`, temperature `?LBOT` and servo `?LBOSS`; apply the operator's readiness criteria |
| `writer.set_lbo_heater(enabled=False)` | Start LBO cool down | Follow the manual's heater/cooling shutdown procedure |
| `writer.set_lbo_optimization()` | Initiate LBO optimization | Relevant servo/temperature readings; acknowledgment does not establish convergence |
| `writer.set_flash()` | Drop below lasing threshold for etalon recentering | Follow the manual's optical procedure; no dedicated completion getter exists |
| `writer.set_chiller_on()` | Turn on Coherent chiller part 1037271 only | Verify through the approved chiller procedure; no dedicated chiller getter exists |
| `writer.set_chiller_temperature_c(APPROVED_C)` | Store finite temperature, rounded to one decimal | Verify through the chiller procedure; the manual does not define the physical setting range |

`enabled` requires an actual Boolean. The LBO heater controls the nonlinear
crystal heater. Check equipment compatibility before using optional chiller functions.


In [ ]:
# set_lbo_heater: heat
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_lbo_heater(enabled=True)
    print("Heater code:", writer.get_value("?LBOH"))
    print("Temperature:", writer.get_value("?LBOT"), "degrees C")
    print("Servo code:", writer.get_value("?LBOSS"))
else:
    print("Skipped: set_lbo_heater: heat")

In [ ]:
# set_lbo_heater: cool down
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_lbo_heater(enabled=False)
    print("Heater code:", writer.get_value("?LBOH"))
else:
    print("Skipped: set_lbo_heater: cool down")

In [ ]:
# set_lbo_optimization: initiate optimization
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_lbo_optimization()
    print("Servo code:", writer.get_value("?LBOSS"))
    print("Temperature:", writer.get_value("?LBOT"), "degrees C")
    print("Acknowledgment does not establish completion.")
else:
    print("Skipped: set_lbo_optimization: initiate optimization")

In [ ]:
# set_flash: etalon recentering
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_flash()
    pprint(writer.last_transaction)
    print("Verify the optical effect through the operator procedure.")
else:
    print("Skipped: set_flash: etalon recentering")

In [ ]:
# set_chiller_on: compatible Coherent chiller only
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_chiller_on()
    pprint(writer.last_transaction)
    print("Verify at the chiller; no dedicated getter is available.")
else:
    print("Skipped: set_chiller_on: compatible Coherent chiller only")

In [ ]:
# set_chiller_temperature_c: store temperature
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.
APPROVED_C = None

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_chiller_temperature_c(APPROVED_C)
    pprint(writer.last_transaction)
    print("Verify at the chiller; no dedicated getter is available.")
else:
    print("Skipped: set_chiller_temperature_c: store temperature")

## 12. Front-panel setters

These methods represent panel operations. They return `None` after acknowledgment;
inspect the front panel for the intended effect. `set_exit_screen()` exits a panel
screen, not the serial session.

| Independent call | Panel operation |
| --- | --- |
| `writer.set_menu_up()` | MENUUP |
| `writer.set_menu_down()` | MENUDN |
| `writer.set_menu_select()` | SELECT |
| `writer.set_exit_screen()` | EXIT |
| `writer.set_knob_down()` | KNOBDN |
| `writer.set_front_panel_lock(VERIFIED_OPERAND)` | LFP integer operand |

The manual does not establish the lock operand's valid range or unlock mapping.
Do not guess a value or treat it as a Boolean. No dedicated panel-state getter is provided.


In [ ]:
# set_menu_up: front-panel action
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_menu_up()
    pprint(writer.last_transaction)
    print("Inspect the front panel for the intended effect.")
else:
    print("Skipped: set_menu_up: front-panel action")

In [ ]:
# set_menu_down: front-panel action
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_menu_down()
    pprint(writer.last_transaction)
    print("Inspect the front panel for the intended effect.")
else:
    print("Skipped: set_menu_down: front-panel action")

In [ ]:
# set_menu_select: front-panel action
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_menu_select()
    pprint(writer.last_transaction)
    print("Inspect the front panel for the intended effect.")
else:
    print("Skipped: set_menu_select: front-panel action")

In [ ]:
# set_exit_screen: front-panel action
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_exit_screen()
    pprint(writer.last_transaction)
    print("Inspect the front panel for the intended effect.")
else:
    print("Skipped: set_exit_screen: front-panel action")

In [ ]:
# set_knob_down: front-panel action
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_knob_down()
    pprint(writer.last_transaction)
    print("Inspect the front panel for the intended effect.")
else:
    print("Skipped: set_knob_down: front-panel action")

In [ ]:
# set_front_panel_lock: verified integer operand
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.
VERIFIED_OPERAND = None  # Do not infer an unlock/lock mapping.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_front_panel_lock(VERIFIED_OPERAND)
    pprint(writer.last_transaction)
    print("Inspect the front panel; operand meanings require verification.")
else:
    print("Skipped: set_front_panel_lock: verified integer operand")

## 13. Serial-link setters

These change how subsequent communication works; do not use them just to explore
an already working connection.

| Independent call | Behavior |
| --- | --- |
| `writer.set_echo(enabled=True)` / `writer.set_echo(enabled=False)` | Enable / disable instrument echo |
| `writer.set_prompt(enabled=True)` / `writer.set_prompt(enabled=False)` | Enable / disable the prompt and update the controller's expected framing |
| `writer.set_baudrate(VERIFIED_BAUD)` | Validate acknowledgment at the old rate, then update the host baud |

Use actual booleans for `enabled`. The named prompt method handles the protocol's
inverted operands (`PROMPT=0` enables; `PROMPT=1` disables) from the supplied
V-2/V-5/V-6 manual. The separate Single FAP manual has different polarity. Successful prompt changes
apply to their acknowledgment and subsequent replies; rejected changes retain
the previous prompt mode.

Use `set_baudrate()` rather than a raw `B=...` instruction so the computer's baud
changes too. Accepted rates are 1200, 2400, 4800, 9600, 19200, 38400 and 57600.
`get_value("?B")` reads the device baud when communication is established. The
manual leaves baud transition timing unspecified: a device that switches before
replying can produce an uncertain outcome. Never replay a failed baud change.


In [ ]:
# set_echo: on
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_echo(enabled=True)
    pprint(writer.last_transaction)
    print("Subsequent version reply:", repr(writer.get_value("?SV")))
else:
    print("Skipped: set_echo: on")

In [ ]:
# set_echo: off
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_echo(enabled=False)
    pprint(writer.last_transaction)
    print("Subsequent version reply:", repr(writer.get_value("?SV")))
else:
    print("Skipped: set_echo: off")

In [ ]:
# set_prompt: on
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_prompt(enabled=True)
    pprint(writer.last_transaction)
    print("Subsequent version reply:", repr(writer.get_value("?SV")))
else:
    print("Skipped: set_prompt: on")

In [ ]:
# set_prompt: off
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_prompt(enabled=False)
    pprint(writer.last_transaction)
    print("Subsequent version reply:", repr(writer.get_value("?SV")))
else:
    print("Skipped: set_prompt: off")

In [ ]:
# set_baudrate: device and host baud
RUN_THIS_EXAMPLE = False  # Change only this cell to True to send its command.
VERIFIED_BAUD = None

if RUN_THIS_EXAMPLE:
    if writer is None:
        raise RuntimeError("Open the write session in section 9 first.")
    writer.set_baudrate(VERIFIED_BAUD)
    pprint(writer.last_transaction)
    print("Device baud:", writer.get_value("?B"))
else:
    print("Skipped: set_baudrate: device and host baud")

<a id="reference"></a>
# Supporting reference

The catalogue examples below are executable and send no instrument instructions.

## A. Discover available queries and instructions

Use `QUERY_INFO` to find the short query names accepted by `get_value()` and
`get_capture()`. `spec` describes each query's name, unit, parsing kind and code
labels. Looking up metadata opens no port and sends no query.


In [ ]:
from coherent_verdi.controller import QUERY_INFO

print(QUERY_INFO["?D1T"])
print(QUERY_INFO["?LBOSS"].labels)

for query, spec in QUERY_INFO.items():
    print(query, spec.name, spec.unit, spec.kind, spec.labels)

A documented query may be unsupported by a particular firmware; `?DIOS` was
rejected in lesson 3. Do not send every catalogue entry as an automatic survey.

`COMMAND_INFO` describes the canonical text instructions used by setters; its
labels do not establish physical readiness or undocumented operating ranges.


In [ ]:
from coherent_verdi.controller import COMMAND_INFO

for command, spec in COMMAND_INFO.items():
    print(command, spec.name, spec.unit, spec.kind, spec.labels)

## B. Lower-level byte transport

Run the `read()` and query `command()` cell after section 8. Section 10 has
individually enabled `write()` and setting `command()` cells for a prepared writer.

Most developer code can stay with `get_...` and `set_...`. Use byte methods only
when you need the serial representation. Instructions are `bytes` without line
endings; the controller adds CR/LF and consumes the reply CR/LF and enabled
trailing prompt.

| Method | Returned result |
| --- | --- |
| `laser.read(b"?P")` | Payload bytes with echo, line endings and prompt removed |
| `laser.command(b"?SV")` | Full wire reply bytes, including framing |
| `writer.write(b"P=0.25")` | Full wire reply bytes; caller interprets acceptance |

`command()` is the shared transport for queries and settings. Raw operations do
not interpret device-error replies as typed helpers do. `get_value()` converts a
query payload; `set_value()` validates a setting acknowledgment. Named methods
build on those interfaces. Full wire bytes remain in `last_transaction["response"]`.


### Reply and timeout handling

With `prompt_enabled=True`, the controller reads the reply through CR/LF, then
consumes the enabled trailing prompt before sending another instruction. The
instrument prompt must already be enabled; construction sends no echo or prompt
changes. Matching instruction echoes and recognized leading notices are removed
from `read()` payloads while their received bytes remain in transaction evidence.

`TIMEOUT_S` configures serial write and receive waits. A reply line, each leading
notice and the trailing prompt use separate `read_until()` calls with that
configured timeout; there is no combined transaction deadline. No extra pause
is inserted between queries. Restart the kernel after a package update.

With `prompt_enabled=False`, reply CR/LF completes the exchange; use this flag
only for an already prompt-disabled device. Echo handling and the named
`set_echo()` / `set_prompt()` methods remain available.

Up to eight exact leading `min current set`, `level1 set!` or `level0 set!` notice
lines are accepted before the reply, within the 4096-byte response limit. The full
`response` retains those bytes; `read()` and typed helpers exclude them from the
payload. Their physical meaning is not established. Unknown extra lines,
ambiguous replies and missing required prompts stop the session without replay.


## C. Transaction record and error details

`last_transaction` and `get_capture()` use exactly six fields:

| Field | Meaning |
| --- | --- |
| `instruction` | Instruction bytes supplied to the transport |
| `request` | Sent bytes, including line endings |
| `response` | Full consumed wire reply, including partial bytes on failure |
| `payload` | Answer without framing when available; capture always uses bytes |
| `outcome` | Operation result label |
| `error` | Diagnostic text, or `None` |

Byte fields need explicit encoding before saving as JSON. `last_transaction` is
`None` before the first attempt. `error.transaction` retains evidence when available.

| Outcome | Meaning |
| --- | --- |
| `received` | Raw operation received a complete frame without interpreting acceptance |
| `ok` / `captured` | Typed query parsed / read-only capture completed |
| `acknowledged` | Setting accepted at protocol level; physical effect still needs verification |
| `rejected` | Complete documented device rejection |
| `failed` / `unknown` | Failed query / uncertain setting that may already have executed |

Catch `DeviceError` before `VerdiError`. A rejected reply naming the wrong query
also retires the session because request/reply alignment is uncertain. Communication
or parsing failures retire the controller. There is no automatic retry or failure-latch reset.


## D. Use a context manager in a script

A context manager opens the port on entry and releases it on exit, including
when the body raises. It sends no shutdown commands. Release the notebook's
connections before starting this separate session; the enabled cell does this
explicitly. If you tested baud or prompt changes, update this constructor to
match the accepted device settings before running it.


In [ ]:
RUN_THIS_EXAMPLE = False  # Optional separate read-only session.

if RUN_THIS_EXAMPLE:
    laser.disconnect()
    if writer is not None:
        writer.disconnect()
    with VerdiController(
        PORT,
        model=MODEL,
        baudrate=BAUDRATE,
        timeout_s=TIMEOUT_S,
        prompt_enabled=True,
    ) as reader:
        reader.purge_buffers()  # Explicitly once, before the first query.
        power_w = reader.get_power_w()
        status = reader.get_status()
    pprint(status)
else:
    print("Context-manager example skipped.")

The notebook separates opening from first communication so you can inspect each step.
Properties such as `model`, `is_simulated`, `allow_writes`, `power_limit_w`,
`failed` and `last_transaction` can be inspected without sending instrument queries.


<a id="finish"></a>
# Handle errors and finish


## 14. Inspect transaction evidence after a failure

| Exception | Meaning for application code |
| --- | --- |
| `ValueError` | Local argument validation refused the call before I/O |
| `PermissionError` | Local permission check refused the call before I/O |
| `DeviceError` | Documented device rejection; a correctly aligned rejection leaves the session usable |
| Other communication/parsing `VerdiError` | Session is retired; stop, preserve evidence and disconnect |

Catch `DeviceError` before its parent `VerdiError`. This beginner tour stops after
an unexpected device reply as well; do not continue by guessing its meaning.

`last_transaction` is a copy of the most recent exchange, not a session log.
Reading it sends nothing. `response` preserves full consumed wire bytes, including
partial bytes on failure. Local refusals preserve the previous record. Save your
connection settings separately. Appendix C explains the record's six fields and outcomes.

Run this evidence cell after an earlier failure once `laser` exists, then run
Disconnect. A failed setter may already have changed the instrument: do not retry it.


In [27]:
evidence = laser.last_transaction
pprint(evidence)
print("Session failed:", laser.failed)
if writer is not None:
    print("Writer transaction:")
    pprint(writer.last_transaction)
    print("Writer failed:", writer.failed)

{'error': None,
 'instruction': b'?SV',
 'outcome': 'received',
 'payload': None,
 'request': b'?SV\r\n',
 'response': b'6.05\r\nVerdi V-2>'}
Session failed: False
Writer transaction:
{'error': "VerdiError: prompt timed out or unexpected: b'prom#1 "
          'hours=2390123.652049\\r\\nincr = 0.111185\\r\\nprom#3 '
          "hours=2695.019283\\r\\n\\r\\nVerdi V-2>'; full response: b'level0 "
          'set!\\r\\nincr = 0.111174\\r\\nprom#1 '
          'hours=2390123.652049\\r\\nincr = 0.111185\\r\\nprom#3 '
          "hours=2695.019283\\r\\n\\r\\nVerdi V-2>'",
 'instruction': b'S=0',
 'outcome': 'unknown',
 'payload': None,
 'request': b'S=0\r\n',
 'response': b'level0 set!\r\nincr = 0.111174\r\nprom#1 hours=2390123.65204'
             b'9\r\nincr = 0.111185\r\nprom#3 hours=2695.019283\r\n\r\nVerdi V-'
             b'2>'}
Writer failed: True


## 15. Disconnect

`disconnect()` releases the serial port and sends no instrument instructions.
Run both cleanup cells below; the first releases the optional writer if created.
No output is expected. It does not close the shutter, stop emission or perform
heater/cooling shutdown. Kernel interruption and closing the notebook are also
not emergency stops; use the operator's physical procedure when needed.

A healthy controller can reconnect. A failed controller must be retired; purging,
reopening or simply making a new object does not establish that the physical link is clean.


In [28]:
# Also run this after an optional writer example fails.
if writer is not None:
    writer.disconnect()

In [29]:
laser.disconnect()